## 1. import libraries

In [1]:
import numpy as np
import random
from IPython.display import display, clear_output
import time

## 2. Environment Setup

In [2]:
# Grid size
ROWS = 10
COLS = 10

# Rewards grid (everything 0 except special cells)
rewards = np.zeros((ROWS, COLS))

# Set special grid rewards
rewards[1, 7] = 5
rewards[4, 2] = 3
rewards[2, 9] = -2

# Starting position (row, col)
start_state = (3, 3)

# Actions: Up, Down, Left, Right
actions = {
    0: (-1, 0),  # Up
    1: (1, 0),   # Down
    2: (0, -1),  # Left
    3: (0, 1)    # Right
}

def is_terminal(state):
    r, c = state
    return rewards[r, c] != 0

In [3]:
rewards

array([[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  5.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -2.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  3.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.]])

In [4]:
start_state

(3, 3)

In [5]:
actions

{0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}

In [6]:
is_terminal((0,0)) 

np.False_

In [7]:
is_terminal((2,2)) 

np.False_

In [8]:
is_terminal((4,2))

np.True_

In [9]:
is_terminal((1,7))

np.True_

## 3. Q table initialization

In [10]:
Q = np.zeros((ROWS, COLS, 4))  # 4 actions per state

## 4. Helper: step function

In [11]:
def step(state, action):
    r, c = state
    dr, dc = actions[action]
    nr, nc = r + dr, c + dc

    # Stay inside grid
    nr = max(0, min(ROWS - 1, nr))
    nc = max(0, min(COLS - 1, nc))

    reward = rewards[nr, nc] - 0.1
    done = is_terminal((nr, nc))

    return (nr, nc), reward, done

## 5. Training parameters

In [12]:
alpha = 0.1     # learning rate
gamma = 0.99    # discount factor
epsilon = 0.1   # exploration
episodes = 200000

## 6. Q learning loop

In [13]:
for ep in range(episodes):
    epsilon = max(0.01, epsilon * 0.999)
    state = start_state

    for _ in range(200):  # max steps per episode
        r, c = state

        # Choose action (epsilon-greedy)
        if random.random() < epsilon:
            action = random.choice(list(actions.keys()))
        else:
            action = np.argmax(Q[r, c])

        next_state, reward, done = step(state, action)
        nr, nc = next_state

        # Q-learning update
        Q[r, c, action] = Q[r, c, action] + alpha * (
            reward + gamma * np.max(Q[nr, nc]) - Q[r, c, action]
        )

        state = next_state

        if done:
            break

print("Training completed!")

Training completed!


## 7. Display the elarning policy

In [14]:
symbols = {0: "↑", 1: "↓", 2: "←", 3: "→"}

policy = np.empty((ROWS, COLS), dtype=object)

for r in range(ROWS):
    for c in range(COLS):
        if is_terminal((r, c)):
            policy[r, c] = f"({int(rewards[r,c])})"
        else:
            policy[r, c] = symbols[np.argmax(Q[r, c])]

policy

array([['←', '→', '→', '↑', '←', '←', '↑', '↓', '↑', '↑'],
       ['↓', '→', '↓', '↓', '←', '←', '→', '(5)', '↑', '↑'],
       ['→', '→', '↓', '↓', '←', '→', '←', '↑', '↑', '(-2)'],
       ['→', '→', '↓', '↓', '←', '←', '↑', '↑', '↑', '↑'],
       ['→', '→', '(3)', '←', '←', '←', '→', '↑', '↑', '↑'],
       ['↓', '→', '↑', '↑', '↑', '→', '↓', '↑', '↑', '↑'],
       ['↓', '↓', '↑', '↓', '↓', '↓', '↑', '↑', '↑', '↑'],
       ['↑', '↑', '↑', '↑', '↑', '↑', '↑', '↑', '↑', '↑'],
       ['↑', '↑', '↑', '↑', '↑', '↑', '↑', '↑', '↑', '↑'],
       ['↑', '↑', '↑', '↑', '↑', '↑', '↑', '↑', '↑', '↑']], dtype=object)

## 8. Run one episode with learned Q value

In [15]:
state = start_state
path = [state]

for _ in range(30):  # limit
    r, c = state
    action = np.argmax(Q[r, c])
    next_state, reward, done = step(state, action)
    path.append(next_state)
    state = next_state
    if done:
        break

path


[(3, 3), (4, 3), (4, 2)]

## 9. Show path visually

In [16]:
grid_display = np.full((ROWS, COLS), ".", dtype=object)

# Mark reward cells
grid_display[1, 7] = "5"
grid_display[4, 2] = "3"
grid_display[2, 9] = "-2"

# Mark visited path
for (r, c) in path:
    if grid_display[r, c] == ".":
        grid_display[r, c] = "*"

# Start and end markers
sr, sc = start_state
grid_display[sr, sc] = "S"
er, ec = path[-1]
if (er, ec) != start_state and grid_display[er, ec] == "*":
    grid_display[er, ec] = "E"

grid_display




array([['.', '.', '.', '.', '.', '.', '.', '.', '.', '.'],
       ['.', '.', '.', '.', '.', '.', '.', '5', '.', '.'],
       ['.', '.', '.', '.', '.', '.', '.', '.', '.', '-2'],
       ['.', '.', '.', 'S', '.', '.', '.', '.', '.', '.'],
       ['.', '.', '3', '*', '.', '.', '.', '.', '.', '.'],
       ['.', '.', '.', '.', '.', '.', '.', '.', '.', '.'],
       ['.', '.', '.', '.', '.', '.', '.', '.', '.', '.'],
       ['.', '.', '.', '.', '.', '.', '.', '.', '.', '.'],
       ['.', '.', '.', '.', '.', '.', '.', '.', '.', '.'],
       ['.', '.', '.', '.', '.', '.', '.', '.', '.', '.']], dtype=object)